# 02 · Prefill vs Decode：两个阶段，两种瓶颈

这一章回答一个面试高频问题：

> **为什么说 prefill 是算力密集、decode 是显存带宽密集？**

背结论没用，面试官会追问"那 batch=1 的 decode 能跑多快"。这一章把公式推一遍，再实测验证。

In [ ]:
# ===== 引导单元：环境检查 + 测量工具 + MiniGPT（每章自带，直接运行）=====
# 说明：本单元在每个 notebook 里都有一份完整副本，目的是让任何一个 notebook
#       都能在 Colab 里零配置独立运行。想改模型结构，请改 tools/build_notebooks.py
#       里的 SETUP_CODE，然后重跑编译脚本。
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# MiniGPT 只有 2700 万参数，用 float16 跑在 GPU 上；CPU 上 float16 很慢，用 float32
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32


def sync():
    """GPU 是异步执行的，计时前必须同步，否则测到的是下发时间不是执行时间。"""
    if DEVICE == "cuda":
        torch.cuda.synchronize()


def bench(fn, warmup=3, iters=10):
    """返回单次调用的平均耗时（毫秒）。warmup 用来排除首次 kernel 编译等开销。"""
    for _ in range(warmup):
        fn()
    sync()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    sync()
    return (time.perf_counter() - t0) / iters * 1000.0


def peak_mem_mb():
    """当前 CUDA 峰值显存占用（MB）。"""
    if DEVICE != "cuda":
        return 0.0
    return torch.cuda.max_memory_allocated() / 1024 ** 2


def reset_peak():
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()


class Config:
    def __init__(self, vocab_size=50257, block_size=1024, n_layer=4, n_head=6, n_embd=384):
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.head_dim = n_embd // n_head


class CausalSelfAttention(nn.Module):
    """因果自注意力，支持 KV cache。

    past_kv 传入历史的 (k, v)，本步只为新 token 计算 Q/K/V，然后拼在历史后面。
    返回 (输出, 更新后的 (k, v))，其中 k/v 的 shape 是 (B, n_head, 总长度, head_dim)。
    """

    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.head_dim = cfg.head_dim
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)

    def forward(self, x, past_kv=None, attn_mask=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)

        S = k.size(2)  # 总长度 = 历史 + 本步新增
        if attn_mask is None:
            # 默认因果掩码：本步第 i 个 query 的绝对位置是 S-T+i，只能看见 <= 它的 key
            mask = torch.ones(T, S, device=x.device).tril(diagonal=S - T).bool()
        else:
            # 外部传入的掩码，用于一个 batch 里混合不同进度的序列（第 04、06 章）
            mask = attn_mask
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y), (k, v)


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=False)

    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x, past_kv=None, attn_mask=None):
        h, present = self.attn(self.ln_1(x), past_kv, attn_mask)
        x = x + h
        x = x + self.mlp(self.ln_2(x))
        return x, present


class MiniGPT(nn.Module):
    """极简 GPT，结构与 Llama 同源：pre-norm + 因果注意力 + 4 倍扩张 MLP + 权重共享。

    与 Llama 的两处差异：
      - 用可学习位置编码代替 RoPE（简化实现，不影响调度实验的结论）
      - 没有 GQA（本仓库是 MHA，第 03 章会手工比较两者的 KV cache 大小）
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.wpe = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight  # 权重共享，省一份 embedding 参数

        def init(m):
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)

        self.apply(init)

    def forward(self, idx, past_kvs=None, pos_offset=0, attn_mask=None):
        """idx: (B, T) 的 token id。

        past_kvs: 长度等于层数的列表，每项是 (k, v)；None 表示从零开始（prefill）。
        pos_offset: 本次输入的第一个 token 的绝对位置。传 int 表示整个 batch 用同一个
                    偏移；传 shape (B,) 的张量表示每条序列各用各的偏移——当 batch 里
                    混合了不同进度的请求时必须这样传。
        attn_mask: 可选的自定义注意力掩码，用于屏蔽填充位。
        """
        B, T = idx.shape
        if torch.is_tensor(pos_offset):
            pos = pos_offset.view(B, 1) + torch.arange(T, device=idx.device)[None, :]
        else:
            pos = torch.arange(pos_offset, pos_offset + T, device=idx.device)[None, :].expand(B, T)
        x = self.wte(idx) + self.wpe(pos)

        presents = []
        for i, blk in enumerate(self.blocks):
            past = None if past_kvs is None else past_kvs[i]
            x, present = blk(x, past, attn_mask)
            presents.append(present)
        return self.lm_head(self.ln_f(x)), presents

    @property
    def n_params(self):
        return sum(p.numel() for p in self.parameters())


def build_model(seed=0, device=DEVICE, dtype=DTYPE, **kw):
    torch.manual_seed(seed)
    cfg = Config(**kw)
    model = MiniGPT(cfg).to(device=device, dtype=dtype)
    return model.eval()


@torch.no_grad()
def generate_naive(model, idx, max_new_tokens):
    """不用 KV cache：每一步都把完整序列重新算一遍（O(n^2) 重算）。"""
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -model.cfg.block_size:])
        idx = torch.cat([idx, logits[:, -1].argmax(-1, keepdim=True)], dim=1)
    return idx


@torch.no_grad()
def generate_cached(model, idx, max_new_tokens):
    """用 KV cache：prompt 只 prefill 一次，之后每步只喂 1 个 token。"""
    logits, past = model(idx)
    nxt = logits[:, -1].argmax(-1, keepdim=True)
    out = [nxt]
    pos = idx.size(1)
    for _ in range(max_new_tokens - 1):
        logits, past = model(nxt, past_kvs=past, pos_offset=pos)
        pos += 1
        nxt = logits[:, -1].argmax(-1, keepdim=True)
        out.append(nxt)
    return torch.cat([idx] + out, dim=1)


def kv_bytes(n_layer, n_kv_head, head_dim, seq_len, batch=1, dtype_bytes=2):
    """KV cache 字节数。注意是 2（K 和 V 各一份）。"""
    return 2 * n_layer * n_kv_head * head_dim * seq_len * batch * dtype_bytes


print(f"引导单元加载完成 | device={DEVICE} dtype={DTYPE} torch={torch.__version__}")
# ===== 引导单元结束 =====

## 一、理论：一个 token 要搬多少字节，做多少次浮点运算

**前向传播的 FLOPs**：每个参数参与一次乘法加一次加法，所以处理 `T` 个 token 大约是

```
FLOPs ≈ 2 × N_params × T
```

**需要搬运的字节数**：权重必须从显存读进计算单元，

```
Bytes ≈ N_params × dtype_bytes
```

两者相除就是**算术强度**（arithmetic intensity），单位是 FLOP/byte：

| 阶段 | token 数 T | 算术强度 | 归属 |
|---|---|---|---|
| Prefill | T = prompt 长度 | `2NT / 2N` = **T** | T=512 时是 512 → 远高于卡的比值 → **compute-bound** |
| Decode | T = 1 | `2N / 2N` = **1** | 远低于卡的比值 → **memory-bound** |

判断标准：**算术强度 > 卡的算力带宽比 → compute-bound，反之 memory-bound**。

In [ ]:
def card_ratio(spec):
    """卡的算力带宽比（FLOP/byte）。"""
    if not spec["bw_gbps"]:
        return float("nan")
    return spec["fp16_tflops"] * 1e12 / (spec["bw_gbps"] * 1e9)


def forward_flops(n_params, n_tokens):
    return 2 * n_params * n_tokens


def decode_ceiling(n_params, bw_gbps, dtype_bytes=2, batch=1):
    """decode 的理论 token 速率上限。

    每生成一步要读一遍全部权重（N × dtype_bytes 字节），带宽决定了每秒能走多少步。
    batch 个请求共享同一次权重读取，所以吞吐随 batch 线性增长（直到算力或 KV 带宽成为新瓶颈）。
    """
    bytes_per_step = n_params * dtype_bytes
    return bw_gbps * 1e9 * batch / bytes_per_step


ratio = card_ratio(SPEC)
print(f"你的卡：{SPEC['name']}")
print(f"  算力带宽比 = {ratio:.0f} FLOP/byte\n")
print("各阶段算术强度 vs 卡片比值：")
for label, ai in [("prefill T=128", 128), ("prefill T=1024", 1024), ("decode batch=1", 1)]:
    verdict = "compute-bound" if ai > ratio else "memory-bound"
    print(f"  {label:16s} AI={ai:>6}  → {verdict}")

## 二、先算一个真实模型的 decode 天花板

公式摆出来就要拿真模型验证。Llama-3-8B 在 T4 上，batch=1 每秒最多能吐多少 token？

In [ ]:
def human(n):
    for unit in ["", "K", "M", "B"]:
        if abs(n) < 1000:
            return f"{n:.1f}{unit}"
        n /= 1000
    return f"{n:.1f}T"


print(f"{'模型':<18}{'参数量':>10}{'权重占用':>12}{'decode 上限(batch=1)':>22}")
print("-" * 64)
for name, params in [("MiniGPT", model.n_params), ("Llama-3-8B", 8.03e9), ("Qwen2.5-7B", 7.62e9)]:
    ceiling = decode_ceiling(params, SPEC["bw_gbps"])
    print(f"{name:<18}{human(params):>10}{params * 2 / 1024 ** 3:>10.1f}GB{ceiling:>20,.0f} t/s")

print()
print("这张表值得记住：")
print("  大模型在单卡上的 decode 速度，被显存带宽死死卡住。")
print("  想更快只有三条路：换带宽更大的卡、量化把权重变小、或者一次喂多个请求摊薄权重读取。")

## 三、实测 prefill：算力利用率能到多少

让序列长度从 128 涨到 1024，看达成的 TFLOPS 是否随序列变长而提升。

In [ ]:
print(f"{'batch':>6}{'seq':>7}{'耗时(ms)':>12}{'吞吐(t/s)':>14}{'达成TFLOPS':>14}{'MFU':>9}")
print("-" * 64)

for B in [1, 4, 16]:
    for T in [128, 512, 1024]:
        idx = torch.randint(0, model.cfg.vocab_size, (B, T), device=DEVICE)
        ms = bench(lambda: model(idx), warmup=3, iters=10)
        tps = B * T / (ms / 1000)
        tflops = forward_flops(model.n_params, B * T) / (ms / 1000) / 1e12
        mfu = tflops / SPEC["fp16_tflops"] * 100 if SPEC["fp16_tflops"] else float("nan")
        print(f"{B:>6}{T:>7}{ms:>12.1f}{tps:>14,.0f}{tflops:>14.2f}{mfu:>8.2f}%")

print()
print("观察两点：")
print("  1. MFU 大概率很低（个位数百分点）。MiniGPT 太小，GPU 还没热身就结束了——")
print("     这说明 MFU 这个指标对大模型才有意义，小模型上瓶颈是 kernel 启动开销。")
print("  2. 序列越长、batch 越大，效率通常越好，因为矩阵乘法的并行度更高。")

## 四、实测 decode：为什么必须做 batching

这是本章最重要的一段实验。固定上下文长度，只改 batch，看吞吐怎么变。

In [ ]:
@torch.no_grad()
def decode_bench(model, batch, ctx_len, steps=32):
    """返回 (token/s, 每步毫秒)。prefill 不计入计时。"""
    idx = torch.randint(0, model.cfg.vocab_size, (batch, ctx_len), device=DEVICE)
    logits, past = model(idx)
    nxt = logits[:, -1].argmax(-1, keepdim=True)
    pos = ctx_len

    sync()
    t0 = time.perf_counter()
    for _ in range(steps):
        logits, past = model(nxt, past_kvs=past, pos_offset=pos)
        pos += 1
        nxt = logits[:, -1].argmax(-1, keepdim=True)
    sync()
    dt = (time.perf_counter() - t0) / steps
    return batch / dt, dt * 1000


CTX = 128
ceiling1 = decode_ceiling(model.n_params, SPEC["bw_gbps"], batch=1)

print(f"上下文长度 {CTX}，连续 decode 32 步")
print(f"理论带宽上限(batch=1) = {ceiling1:,.0f} token/s\n")
print(f"{'batch':>6}{'吞吐(t/s)':>14}{'每步(ms)':>12}{'占带宽上限':>14}")
print("-" * 48)

for B in [1, 4, 16, 64]:
    tps, ms_step = decode_bench(model, B, CTX)
    pct = tps / (ceiling1 * B) * 100
    print(f"{B:>6}{tps:>14,.0f}{ms_step:>12.2f}{pct:>13.1f}%")

print()
print("这里会出现一个反直觉的现象：")
print("  batch=1 时，达成率可能只有百分之几——GPU 大部分时间在等 kernel 下发，而不是在搬数据。")
print("  随着 batch 增大，同一份权重被更多请求摊薄，达成率快速逼近上限。")
print()
print("这就是 continuous batching 存在的全部理由：")
print("  单请求的 decode 根本无法喂饱 GPU，必须把多个请求拼在一起跑。")

## 五、把结论整理成面试话术

问你"prefill 和 decode 有什么区别"，按这个结构答：

1. **计算形态不同**：prefill 一次处理整段 prompt，是大的稠密 GEMM，算术强度等于序列长度；decode 一次一个 token，算术强度约等于 1。
2. **瓶颈不同**：前者受算力限制（看 MFU 利用率），后者受显存带宽限制（看带宽利用率）。判断依据是算术强度和卡片算力带宽比的比较。
3. **工程含义不同**：因为 decode 是访存密集且单请求喂不饱 GPU，所以要 continuous batching；因为 prefill 吃算力且会长时间占住 GPU，长 prefill 会阻塞其他请求，所以要 chunked prefill（第 06 章）。
4. **可验证**：给他一个具体数字——比如 Llama-3-8B 在 T4 上 batch=1 的理论 decode 上限约为 20 token/s。

**作业**

1. 把 `CTX` 从 128 改到 1024，重跑 decode 实验。随着上下文变长，KV cache 的读取量变大，达成率会怎么变？
2. 用 `decode_ceiling` 算一下：如果换成 H100（3350 GB/s），Llama-3-8B 的 batch=1 上限是多少？和 T4 差几倍？
3. 思考题：为什么增大 batch 能提升吞吐，但**不能降低单请求的延迟**？这对服务设计意味着什么？

**下一章**：decode 阶段的 KV cache 到底占多少显存，一个 80G 的卡能同时服务多少请求。